# 00 – Setup and Data Loading
**IT3091 Machine Learning · Group 2026-DS-02 · Bank Marketing**

Purpose of this notebook:
1. Load the shared project configuration (`src/config.py`).
2. Verify the dataset fingerprint (SHA-256) so every member uses the identical file.
3. Load the data and run basic sanity checks (shape, columns, types, target).

Full EDA is done in `01_eda.ipynb` (M1).

## 1. Imports and shared configuration

In [1]:
import sys, hashlib
from pathlib import Path

import pandas as pd
import numpy as np

# Make src/ importable whether the notebook is run from notebooks/ or the project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config

print("Python       :", sys.version.split()[0])
print("pandas       :", pd.__version__)
print("Raw file     :", config.RAW_FILE.relative_to(config.PROJECT_ROOT))
print("Random state :", config.RANDOM_STATE)

Python       : 3.13.7
pandas       : 2.3.2
Raw file     : data\raw\bank-additional-full.csv
Random state : 42


## 2. Verify dataset fingerprint

In [2]:
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

file_hash = sha256_of(config.RAW_FILE)
print("SHA-256 :", file_hash)
assert file_hash == config.RAW_FILE_SHA256, "Dataset file differs from the agreed version!"
print("Fingerprint OK - this is the agreed dataset file.")

SHA-256 : 74adfc578bf77a7ff4bb1ba4a9f8709d9e3c6907342959c2c8416847e0afb4d8
Fingerprint OK - this is the agreed dataset file.


## 3. Load the data
The file uses `;` as the separator.

In [3]:
df = pd.read_csv(config.RAW_FILE, sep=config.CSV_SEPARATOR)
assert df.shape == config.EXPECTED_SHAPE, f"Unexpected shape {df.shape}"
print("Shape:", df.shape)
df.head()

Shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 4. Columns, types and feature groups

In [4]:
groups = {
    "client": config.CLIENT_FEATURES,
    "contact": config.CONTACT_FEATURES,
    "campaign": config.CAMPAIGN_FEATURES,
    "economic": config.ECONOMIC_FEATURES,
    "target": [config.TARGET],
}
col_group = {c: g for g, cols in groups.items() for c in cols}
assert set(col_group) == set(df.columns), "Config feature groups do not match the file columns"

overview = pd.DataFrame({
    "group": [col_group[c] for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "n_unique": df.nunique().values,
    "example": df.iloc[0].values,
}, index=df.columns)
overview

,group,dtype,n_unique,example
age,client,int64,78,56
job,client,object,12,housemaid
marital,client,object,4,married
education,client,object,8,basic.4y
default,client,object,3,no
housing,client,object,3,no
loan,client,object,3,no
contact,contact,object,2,telephone
month,contact,object,10,may
day_of_week,contact,object,5,mon


## 5. Target variable

In [5]:
target_counts = df[config.TARGET].value_counts()
target_pct = (df[config.TARGET].value_counts(normalize=True) * 100).round(2)
pd.DataFrame({"count": target_counts, "percent": target_pct})

,count,percent
y,,
no,36548,88.73
yes,4640,11.27


## 6. Quick data-quality flags
Recorded here only as pointers for EDA (M1) and preprocessing (M2); no changes are made to the data.

In [6]:
cat_cols = df.select_dtypes(exclude="number").columns
flags = {
    "blank (NaN) cells": int(df.isna().sum().sum()),
    "duplicate rows": int(df.duplicated().sum()),
    "rows with pdays == 999 (not previously contacted)": int((df["pdays"] == config.PDAYS_NOT_CONTACTED).sum()),
}
print(pd.Series(flags).to_string())
print()
unknown_counts = (df[cat_cols] == config.MISSING_TOKEN).sum()
print("'unknown' values per categorical column:")
print(unknown_counts[unknown_counts > 0].sort_values(ascending=False).to_string())

blank (NaN) cells                                        0
duplicate rows                                          12
rows with pdays == 999 (not previously contacted)    39673



'unknown' values per categorical column:
default      8597
education    1731
housing       990
loan          990
job           330
marital        80


## 7. Summary
- The dataset fingerprint matches the agreed file (decision **D1**).
- 41,188 rows × 21 columns; the target `y` is imbalanced (≈11% `yes`).
- No blank cells, but missing values are coded as `"unknown"` in several categorical columns.
- 12 duplicate rows and the `pdays = 999` placeholder need handling in preprocessing.
- `duration` is flagged as a leakage feature (decision **D3**).

**Next:** `01_eda.ipynb` – data dictionary and exploratory data analysis (M1).